### 0. Install packages

In [7]:
# !pip install --upgrade pip
# !conda install -c anaconda requests -y
# !pip install pandas
# !pip install selenium
# !pip install chromadb sentence-transformers
# !pip install openai
# !pip install bs4
# !pip install lxml

  Obtaining dependency information for lxml from https://files.pythonhosted.org/packages/77/d5/becbe1e2569b474a23f0c672ead8a29ac50b2dc1d5b9de184831bda8d14c/lxml-6.0.2-cp311-cp311-macosx_10_9_universal2.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 32.8 MB/s eta 0:00:0000:0100:01


In [5]:
import sys
import time
import os 
import tempfile
from dotenv import load_dotenv 

import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

import tiktoken
import chromadb

import json
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import openai


pd.set_option("display.max_rows", None)

### 1. Scrape CC-related Links on the Commbank Website

1a. Scrape the sitemap for static HTML links

In [8]:
# Fetch the sitemap
sitemap = 'https://www.commbank.com.au/sitemap.xml'
response = requests.get(sitemap)
soup = BeautifulSoup(response.content, 'xml')

# Find all <loc> tags and filiter URLs containing 'credit-card'
urls = [loc.text for loc in soup.find_all('loc') 
        if 'credit-card' in loc.text 
            and 'news' not in loc.text
            and 'eform' not in loc.text
            and 'tools' not in loc.text] # Excludinging urls containing 'news'/ 'eform'/ 'tools'

# Output the urls
url_list = []
for url in urls:
    url_list.append(url)

print(len(url_list))

FeatureNotFound: Couldn't find a tree builder with the features you requested: xml. Do you need to install a parser library?

In [3]:
# Save the output to a text file to examine 
with open("cc_urls.txt", "w") as f:
    for url in url_list:
        f.write(str(url) + "\n")

print("Saved to output.txt. Open the file to see all items.")

NameError: name 'url_list' is not defined

1b. Scrape dynamically rendered content on the 'Manage' page

In [5]:
# Set up Selenium WebDriver 
options = webdriver.ChromeOptions()
options.add_argument("--headless") #Run without opening a browser window
driver = webdriver.Chrome(options=options)


mng_url = "https://www.commbank.com.au/credit-cards/manage.html"
driver.get(mng_url)

# Extract <a> tags using Selenium
slnm_links = [a.get_attribute("href") for a in driver.find_elements(By.TAG_NAME, "a") if a.get_attribute("href")]

# Extract <a> tags using BeautifulSoup (static HTML)
soup = BeautifulSoup(driver.page_source, "html.parser")
bs4_links = [a["href"] for a in soup.find_all("a", href=True)]

# Identify dynamically generated links 
dyno_links = set(slnm_links) - set(bs4_links)
dyno_urls = sorted(dyno_links) # Turn into a sorted list

# Save dyno links to txt
with open("dyno_urls.txt", "w") as f:
    for url in dyno_urls:
        f.write(str(url) + "\n")

driver.quit()


63

83

71

75

134

...

108

86

76

55

78

In [6]:
# Remove irrelevant links 
dyno_urls = [url for url in dyno_urls 
             if '/about-us' not in str(url)
                and '/contact-us' not in str(url)
                and '/travel' not in str(url)
                and '.pdf' not in str(url)
                and 'footer' not in str(url)
                and 'html#' not in str(url)]

# Save dyno links to txt
with open("dyno_urls.txt", "w") as f:
    for url in dyno_urls:
        f.write(str(url) + "\n")

92

88

110

98

87

...

54

81

95

108

86

Combine the two lists and dedup

In [7]:
# Merge the two lists
merge_list = url_list + dyno_urls
merge_list_dedup = sorted(set(merge_list))

print(len(merge_list_dedup))


102


In [8]:
# Manual clean up - outdated link/ manual dedup

rmv = ['https://www.commbank.com.au/articles/credit-cards/what-makes-up-your-credit-card-balance.html?ei=paying_repay'
    , 'https://www.commbank.com.au/business/business-credit-cards/business-awards-credit-cards.html'
    , 'https://www.commbank.com.au/business/business-credit-cards/compare.html'
    , 'https://www.commbank.com.au/business/business-credit-cards/card-selector.html'
    , 'https://www.commbank.com.au/commbank-yello.html'
    , 'https://www.commbank.com.au/credit-cards.html'
    , 'https://www.commbank.com.au/business/business-credit-cards.html'
    , 'https://www.commbank.com.au/credit-cards/credit-card-offers.html'
    , 'https://www.commbank.com.au/credit-cards/credit-cards-offers/expired-offers.html'
    , 'https://www.commbank.com.au/credit-cards/diamond.html'
    , 'https://www.commbank.com.au/credit-cards/low-fee-gold.html'
    , 'https://www.commbank.com.au/credit-cards/low-rate-gold.html'
    , 'https://www.commbank.com.au/credit-cards/commbank-essentials.html'
    , 'https://www.commbank.com.au/credit-cards/manage/switch-card.html?ei=switch'
    , 'https://www.commbank.com.au/credit-cards/flightcentre.html'
    , 'https://www.commbank.com.au/credit-cards/myer.html'
    , 'https://www.commbank.com.au/credit-cards/platinum.html'
    , 'https://www.commbank.com.au/digital-banking/lock-block-limit-your-credit-card.html?ei=control_lbl'
    , 'https://www.commbank.com.au/digital-banking/lock-block-limit-your-credit-card.html'
    , 'https://www.commbank.com.au/support.html?ei=help_faqs']

master_list = [url for url in merge_list_dedup if url not in rmv]
print(len(master_list))

82


In [9]:
# Save the master list in  a text file 
with open("master_list.txt", "w") as f:
    for url in master_list:
        f.write(str(url) + "\n")

106

87

92

79

85

...

81

78

95

108

86

### 2. Scrape the content on links in the master list

In [10]:
def scrape_pages(urls):
    """Scrapes dynamically rendered webpages using Safari WebDriver and stores raw content."""
    
    from selenium import webdriver
    from selenium.webdriver.safari.service import Service
    from bs4 import BeautifulSoup
    import time

    service = Service()  # Create a Safari WebDriver service instance
    driver = webdriver.Safari(service=service)  # Start Safari driver

    scraped_data = []  # Store data for all pages
    
    try:
        for url in urls:
            driver.get(url)  # Load the webpage
            time.sleep(3)  # Allow time for JavaScript to render content
            
            # Get HTML and parse it
            soup = BeautifulSoup(driver.page_source, "html.parser")

            # Extract page title (fallback if <title> tag is missing)
            title = soup.title.text.strip() if soup.title else "Untitled Page"

            # Remove unwanted elements
            for tag in soup(["script", "style", "header", "footer", "nav", "aside"]):
                tag.extract()

            # Remove specific <div> elements by class name
            unwanted_classes = ["skip-links-module", "commbank-header", "article-related list parbase", "experiencefragment"]
            for div in soup.find_all("div", class_=unwanted_classes):
                div.extract()

            # Extract cleaned text
            text = soup.get_text(separator="\n")
            text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])  # Remove empty lines
            
            # Store data in a structured format
            scraped_data.append({"url": url, "title": title, "content": text})

    except Exception as e:
        print(f"Error scraping {url}: {e}")

    finally:
        driver.quit()  # Close Safari browser

    return scraped_data


In [11]:
# Test out the function 
urls = master_list[:2]
scrape_pages(urls)

[{'url': 'https://www.commbank.com.au/articles/business/4-ways-to-protect-your-business-from-credit-card-fraud.html',
  'title': '4 ways to protect your business from credit card fraud',
  'content': "4 ways to protect your business from credit card fraud\n4 ways to protect your business from credit card fraud\nCredit card fraud can negatively impact your businesses’ customers, revenue and reputation. Here are some ways to help ensure your business stays safe.\nWith more businesses starting to trade and receive payments online, there are greater opportunities for credit card fraud. You need to be particularly careful when the card is not present in transactions such as phone, email and online orders.\n1. Watch out for warning signs\nOne of these signs may not be cause for alarm, but it is worth paying special attention if one or more of these things happen:\nLarge unusual transactions from unknown buyers. If they’re in your store you could ask for additional identification to make sure

In [12]:
# Loop through all webpages in the master list 
urls = master_list
scraped_data = scrape_pages(urls)

# Check number of links scraped:
print(len(scraped_data))

82


In [13]:
# save scrapted data to a json file
with open("scraped_data.json", "w", encoding='utf-8') as f:
    json.dump(scraped_data, f, ensure_ascii=False, indent=4)

### 3. Chunking and Embedding of the Scrapped Data

3a. Chunking scraped data 

In [121]:
# Define a function for chunking scraped text
def chunking(scraped_data, chunk_size=500, overlap=100):
    """Splits text into overlappig chunks for better retrievel"""

    chunked_data = []

    for entry in scraped_data:
        url = entry["url"]
        title = entry["title"]
        content = entry["content"]

        words = content.split()
        num_words = len(words)

        # create overlapping chunks
        for i in range(0, num_words, chunk_size - overlap):
            chunk = " ".join(words[i:i+chunk_size])
            chunked_data.append({"url": url, "title": title, "content": chunk})
        
    return chunked_data

In [132]:
# Chunking scraped data
chunked_data = chunking(scraped_data)

# Check output - show first 5 chunks for preview
print("Number of chunks returned:", len(chunked_data))

for chunk in chunked_data[:5]:  
    print(f"Title: {chunk['title']}")
    print(f"Chunk Content: {chunk['content'][:200]}...")  # Truncate for display
    print(f"Source: {chunk['url']}")
    print("-" * 50)

Number of chunks returned: 275
Title: 4 ways to protect your business from credit card fraud
Chunk Content: 4 ways to protect your business from credit card fraud 4 ways to protect your business from credit card fraud Credit card fraud can negatively impact your businesses’ customers, revenue and reputation...
Source: https://www.commbank.com.au/articles/business/4-ways-to-protect-your-business-from-credit-card-fraud.html
--------------------------------------------------
Title: 4 ways to protect your business from credit card fraud
Chunk Content: taking your business online? Take my business online Things you should know This article is intended to provide general information of an educational nature only. It does not have regard to the financ...
Source: https://www.commbank.com.au/articles/business/4-ways-to-protect-your-business-from-credit-card-fraud.html
--------------------------------------------------
Title: Things to consider before getting a business credit card - CommBank
Ch

3b. Embed text chunks and store the embeddings in a vector database

In [144]:
import chromadb

# Initialize ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")  # Stores data persistently

# Create a collection (like a table in a database)
collection = client.get_or_create_collection(name="cc_faqs")


In [146]:
# Covert chunks into embeddings
from sentence_transformers import SentenceTransformer

# load the embedding model
embd_model = SentenceTransformer("all-MiniLM-L6-v2")

# Prepare data for storage
for idx, chunk in enumerate(chunked_data):
    embedding = embd_model.encode(chunk["content"]).tolist() # create embedding for chunked text
    metadata = {"url": chunk["url"], 
                "title": chunk["title"]} # store metadata
    
    # Add to ChromDB
    collection.add(
        ids=[str(idx)],
        embeddings=embedding,
        metadatas=[metadata],
        documents=[chunk["content"]]
    )

Insert of existing embedding ID: 0
Add of existing embedding ID: 0
Insert of existing embedding ID: 1
Add of existing embedding ID: 1
Insert of existing embedding ID: 2
Add of existing embedding ID: 2
Insert of existing embedding ID: 3
Add of existing embedding ID: 3
Insert of existing embedding ID: 4
Add of existing embedding ID: 4
Insert of existing embedding ID: 5
Add of existing embedding ID: 5
Insert of existing embedding ID: 6
Add of existing embedding ID: 6
Insert of existing embedding ID: 7
Add of existing embedding ID: 7
Insert of existing embedding ID: 8
Add of existing embedding ID: 8
Insert of existing embedding ID: 9
Add of existing embedding ID: 9
Insert of existing embedding ID: 10
Add of existing embedding ID: 10
Insert of existing embedding ID: 11
Add of existing embedding ID: 11
Insert of existing embedding ID: 12
Add of existing embedding ID: 12
Insert of existing embedding ID: 13
Add of existing embedding ID: 13
Insert of existing embedding ID: 14
Add of existing em

In [147]:
# Preview data stored in the vector DB
collection.peek()

{'ids': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
 'embeddings': array([[ 0.00113211,  0.04056308,  0.00631971, ..., -0.02422256,
          0.00098081, -0.03522873],
        [ 0.0487119 ,  0.04076566, -0.05382952, ..., -0.0819824 ,
         -0.0301461 , -0.00770714],
        [ 0.01172959,  0.03519329,  0.01906572, ...,  0.03777206,
         -0.03426725, -0.05600312],
        ...,
        [-0.04415829, -0.0258555 , -0.10188524, ...,  0.02172532,
         -0.0220574 , -0.0738541 ],
        [ 0.04745823,  0.07806675, -0.0700125 , ..., -0.02444986,
         -0.00642855, -0.11292671],
        [ 0.06615662,  0.01341025, -0.01555859, ...,  0.04744376,
         -0.00190003, -0.09574125]], shape=(10, 384)),
 'documents': ["4 ways to protect your business from credit card fraud 4 ways to protect your business from credit card fraud Credit card fraud can negatively impact your businesses’ customers, revenue and reputation. Here are some ways to help ensure your business stays safe. With

### 4. Implement RAG

In [160]:
# Use enviroment file to set up openai api
load_dotenv()
openai.api_key = os.environ.get("OPENAI_API_KEY")

True

In [171]:
# Define relevant chunk retriever funciton 
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="cc_faqs")

def retrieve_relevant_chunks (query, top_k=5):
    """Retrieve the most relevant document chunks from scrapped FAQs"""
    results = collection.query( # slimiarity search 
        query_texts = [query], n_results=top_k
    )
    return results["documents"][0]

# Test out the function
test_query = "Can I change to a different credit card?"
retrieve_relevant_chunks(test_query)

["Switch your credit card - CommBank Credit Cards / Manage and control your credit card / Switch your credit card Switch your credit card Switch now Easily switch your CommBank credit card Take advantage of the features and benefits CommBank credit cards have to offer and get the card that's right for you. You can switch your existing card to one of our Low Rate, Low Fee or Awards credit cards. When switching, you won’t be eligible for offers advertised to new customers. CommBank Neo customers are unable to switch their card. Check your eligibility You're eligible to switch your credit card if: You currently hold an eligible Commonwealth Bank issued credit card You're the primary cardholder Your current credit limit meets the minimum limit requirements of the card you want to switch to You're an Australian resident or have an Australian address Your account is in order How to switch your CommBank credit card If you have a valid NetBank ID, the easiest way to switch is via our digital S

In [178]:
# Initialize the OpenAI client
client = openai.OpenAI()

def generate_response(query):
    """Use OpenAI GPT-4 Turbo to generate a response using retrieved context."""
    relevant_chunks = retrieve_relevant_chunks(query)
    context = "\n\n".join(relevant_chunks)

    prompt = f"""
    You are an AI assistant that helps customers with credit card inquiries.
    Use the provided context to answer the question accurately.
    
    Context:
    {context}

    Question: {query}
    
    Answer:
    """

    response = client.chat.completions.create(
        model="gpt-4-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

# Test 
query = "What's the interest rate I will be charged?"
response = generate_response(query)
print(response)


To accurately answer the question of what interest rate you will be charged, it's important to know which CommBank credit card product you are asking about, as different cards have different rates. Here are some scenarios based on the provided context:

1. **Platinum Awards and Diamond Awards Cards:**
   - **Interest rate on purchases:** 20.99% p.a.
   - **Interest rate on cash advances:** 21.99% p.a.

2. **Business Gold Awards Card:**
   - **Interest rate on purchases:** 20.74% p.a.
   - **Interest rate on cash advances:** 21.24% p.a.

3. **Low Rate Credit Card (for new applications from 10 December 2024):**
   - **Personalised interest rate on purchases:** Ranges from 10.99% p.a. to 15.99% p.a., depending on factors such as your credit risk score. If you are switching from another CommBank credit card, the standard interest rate is 13.99% p.a.
   - **Interest rate on cash advances:** This rate is not personalized and will be standard across all Low Rate credit cards. This specific ra

In [180]:
# Test 2
query = "Can I change to a different credit card? How do I do it?"
response = generate_response(query)
print(response)

Yes, you can change to a different credit card with CommBank. Here’s how you can do it:

1. **Check Eligibility**: First, ensure you are eligible to switch your card. You must be the primary cardholder of a current eligible CommBank credit card, meet the minimum credit limit requirements for the new card, and your account must be in order. Note that CommBank Neo customers cannot switch their card.

2. **Choose Your New Card**: Log on to NetBank or open the CommBank app, and use the ‘Switch your credit card’ option provided. Compare your existing card with other available options to find the one that fits your needs.

3. **Review Card Information**: Once you have selected your new card, review all the new card information and confirm your mailing address.

4. **Review Terms and Conditions**: It’s important to read through the terms and conditions associated with the new card to understand any changes or requirements.

5. **Confirm Your Switch**: Confirm your switch after agreeing to the

In [181]:
# Test 3
query = "How do I earn Qantas point on my credit card?"
response = generate_response(query)
print(response)

To earn Qantas Points on your CommBank credit card, you have two primary options based on the type of card and program you are enrolled in:

1. **Ultimate Awards Credit Card with Qantas Points**:
   - You can earn up to 1.2 Qantas Points for every $1 you spend on eligible purchases, up to and including $10,000 in a statement period.
   - For any spend over $10,000 within the same statement period, you will then earn 0.2 Qantas Points per $1 spent for the remainder of that period.
   - There is no annual limit to how many Qantas Points you can earn.

2. **Opting-In for Qantas Points with Other CommBank Awards Credit Cards**:
   - For cards under the CommBank Awards program that do not automatically earn Qantas Points, you need to opt-in to start earning Qantas Points.
   - Once opted in, you will earn Qantas Points at a rate determined by the specific card you hold, often converting from CommBank Awards points to Qantas Points. For example, the conversion rate might be 2.5 CommBank Awar